# CZI Cell Analyzer

This notebook discovers CZI channels, creates independent settings for every channel, segments cells from one selected channel, and measures all channels inside the resulting ROIs.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

from cell_analyzer.config import create_default_config, save_yaml
from cell_analyzer.czi_io import inspect_czi
from cell_analyzer.pipeline import run_analysis

## 1. Select the input and output

Use a lower zoom for parameter tuning and `1.0` for final measurements when memory allows.

In [ ]:
CZI_PATH = Path(r"D:\data\example.czi")
OUTPUT_DIR = Path(r"D:\data\cell_analysis_output")
TUNING_ZOOM = 0.25

## 2. Inspect the CZI and create channel settings

In [ ]:
info = inspect_czi(CZI_PATH)
info.to_dict()

In [ ]:
config = create_default_config(info, output_dir=OUTPUT_DIR, zoom=TUNING_ZOOM)
config["channels"]

## 3. Optional direct parameter edits

Repeat the channel block for every detected channel that needs different values.

In [ ]:
config["input"]["segmentation_channel"] = 0
config["segmentation"].update({
    "threshold_method": "otsu",
    "threshold_scale": 0.8,
    "min_area_px": 50,
    "max_area_px": None,
    "clear_border": True,
    "border_exclusion_margin_px": 10,
    "fill_all_holes": True,
    "split_touching": False,
    "min_peak_distance_px": 8,
    "watershed_min_peak_height_px": 0.0,
    "watershed_min_peak_prominence_px": 0.0,
})

config["channels"]["0"].update({
    "gaussian_sigma_px": 1.0,
    "measurement_threshold": {"method": "otsu", "percentile": 95.0},
})

save_yaml(config, "notebook_config.yaml")

## 4. Live parameter tuning

Click **Select CZI files** to choose multiple images. Use the file selector to preview each image, adjust the shared parameters, and click **Run all files** when the settings are satisfactory.

In [ ]:
from cell_analyzer.interactive import launch_batch_tuning_widget

tuner = launch_batch_tuning_widget(config)
tuner

## 5. Inspect batch results

Use the **Run all files** button above, then inspect the aggregate tables.

In [ ]:
batch_result = tuner.batch_result
if batch_result is None:
    raise RuntimeError("Click 'Run all files' and wait for the batch to finish first.")
batch_result

In [ ]:
batch_summary = pd.read_csv(batch_result["files"]["batch_summary_csv"])
measurements = pd.read_csv(batch_result["files"]["combined_measurements_csv"])
display(batch_summary)
print(f"Total ROIs: {batch_result['total_roi_count']} | Measurement rows: {len(measurements)}")
hidden_columns = ["centroid_x_px", "centroid_y_px", "major_axis_length_analysis_px", "minor_axis_length_analysis_px", "eccentricity"]
display(measurements.drop(columns=hidden_columns, errors="ignore"))